<div style="border:1px solid #d8dee9; border-radius:8px; padding:16px 18px; background:#fbfcff; margin-bottom:14px;">
  <h1 style="margin:0 0 6px 0;">Most active TE subfamilies in senescence</h1>
  <p style="margin:0; color:#586069;"><strong>Notebook:</strong> R kernel &nbsp; | &nbsp; exploratory follow-up</p>
</div>


## Pipeline overview

Small follow-up check for the TE-Seq senescence run. I rank TE subfamilies at `k = 3, 5, 10, 20, 50, 100`.

Two ranked TE lists are compared:

- `craTEs activity`: default ranking by sum of absolute condition mean activity scores.
- `TE expression`: use `E_TE-Seq_prep_results.tsv`, keep `gene_or_te == "repeat"`, average per condition, then resolve duplicate TE names with `Max Sum`.

I do this once globally and once for each condition.

---
## 0 · Setup

In [27]:
# Works from the notebook folder and from the repo root.
find_project_dir <- function(start = getwd()) {
  current <- normalizePath(start, winslash = "/", mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "scripts", "craTEs_own_function.R")) &&
        file.exists(file.path(current, "scripts", "most_active_te_subfamilies_f.R")) &&
        dir.exists(file.path(current, "R"))) {
      return(current)
    }
    parent <- dirname(current)
    if (identical(parent, current)) {
      stop("Could not find the CraTEs Senescence project root.", call. = FALSE)
    }
    current <- parent
  }
}

project_dir <- find_project_dir()
setwd(project_dir)
project_dir


[1] "/Users/vinzent/craTEs/CraTEs_Senescence_Analysis"

In [28]:
# Small dependency set; the heavy lifting is in scripts/.
suppressPackageStartupMessages({
  library(data.table)
})

source_project_file <- function(..., envir = .GlobalEnv) {
  path <- file.path(project_dir, ...)
  if (!file.exists(path)) {
    stop("Missing project helper file: ", path, call. = FALSE)
  }
  source(path, local = envir)
}

source_project_file("scripts", "craTEs_own_function.R")
source_project_file("scripts", "most_active_te_subfamilies_f.R")


[1] "sourcing the custom function from crates"


### -- Parameters --

Defaults point to the Hg38 counted senescence run.

In [29]:
k_values <- c(3L, 5L, 10L, 20L, 50L, 100L)

result_dir <- "results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs"
activity_scores_path <- file.path(result_dir, "activity_scores.tsv")

condition_order <- c("PRO", "ESEN", "LSEN")

te_expression_path_candidates <- c(
  "data/Hg38/E_TE-Seq_prep_results.tsv",
  "data/Hg38/GSE280365_resultsdf.tsv"
)
N_path <- "data/Hg38/N_c_100_Hg38_craTEs.tsv"

# Local convenience for the nested working copy used during cleanup.
resolve_local_path <- function(path) {
  if (file.exists(path)) return(path)
  parent_path <- file.path(dirname(project_dir), path)
  if (file.exists(parent_path)) return(parent_path)
  path
}
te_expression_path_labels <- te_expression_path_candidates
te_expression_path_resolved <- vapply(te_expression_path_candidates, resolve_local_path, character(1))
available_te_expression <- which(file.exists(te_expression_path_resolved))
te_expression_path <- if (length(available_te_expression) > 0L) te_expression_path_resolved[[available_te_expression[[1]]]] else te_expression_path_resolved[[1]]
te_expression_path_label <- if (length(available_te_expression) > 0L) te_expression_path_labels[[available_te_expression[[1]]]] else te_expression_path_labels[[1]]
N_path_label <- N_path
N_path <- resolve_local_path(N_path)

# Duplicate handling only; this is not the ranking score.
te_duplicate_method <- "Max Sum"

activity_ranking_metric <- "sum_abs_mean_activity_score"
output_dir <- file.path(result_dir, "exploratory", "most_active_te_subfamilies")
write_incomplete_outputs <- TRUE

data.frame(
  setting = c("result_dir", "activity_scores_path", "te_expression_path", "N_path", "te_duplicate_method", "activity_ranking_metric", "output_dir", "write_incomplete_outputs"),
  value = c(result_dir, activity_scores_path, te_expression_path_label, N_path_label, te_duplicate_method, activity_ranking_metric, output_dir, write_incomplete_outputs),
  stringsAsFactors = FALSE
)


setting,value
<chr>,<chr>
result_dir,results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs
activity_scores_path,results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/activity_scores.tsv
te_expression_path,data/Hg38/E_TE-Seq_prep_results.tsv
N_path,data/Hg38/N_c_100_Hg38_craTEs.tsv
te_duplicate_method,Max Sum
activity_ranking_metric,sum_abs_mean_activity_score
output_dir,results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies
write_incomplete_outputs,TRUE


---
## 1 · craTEs Activity Ranking

The activity score matrix `A` comes from `activity_scores.tsv`. The global ranking is the default sum of absolute condition means.

For each TE subfamily `t`:

$$R_t = |\overline{A}_{t,PRO}| + |\overline{A}_{t,ESEN}| + |\overline{A}_{t,LSEN}|$$

Condition-specific lists use the absolute mean activity in the matching condition.

In [30]:
# Stored craTEs activity scores -> global and condition-specific rankings.
A <- load_activity_scores_matrix(activity_scores_path)
condition_groups <- infer_senescence_condition_groups(colnames(A))
ranking_scopes <- c("global", condition_order)

activity_ranked_by_scope <- stats::setNames(lapply(ranking_scopes, function(scope) {
  rank_activity_te_subfamilies(
    activity_scores = A,
    condition_groups = condition_groups,
    k_values = k_values,
    condition_order = condition_order,
    ranking_metric = activity_ranking_metric,
    scope = scope
  )
}), ranking_scopes)
activity_ranked <- activity_ranked_by_scope$global

activity_top_k <- do.call(rbind, lapply(activity_ranked_by_scope, function(ranked_table) {
  top_k_long_table(
    ranked_table = ranked_table,
    k_values = k_values,
    source = "cis_regulatory_activity",
    score_col = "ranking_score"
  )
}))
rownames(activity_top_k) <- NULL

activity_ranked_display <- activity_ranked
numeric_cols <- vapply(activity_ranked_display, is.numeric, logical(1))
activity_ranked_display[numeric_cols] <- lapply(activity_ranked_display[numeric_cols], round, digits = 4)
head(activity_ranked_display, 20)


,scope,rank,te_subfamily,ranking_metric,ranking_score,signed_activity_score,sum_abs_mean_activity_score,mean_PRO,abs_mean_PRO,mean_ESEN,abs_mean_ESEN,mean_LSEN,abs_mean_LSEN
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,global,1,HERV4_I-int,sum_abs_mean_activity_score,0.1688,NA,0.1688,-0.0996,0.0996,0.0687,0.0687,0.0004,0.0004
2,global,2,LTR13,sum_abs_mean_activity_score,0.1390,NA,0.1390,0.0804,0.0804,-0.0335,0.0335,-0.0252,0.0252
3,global,3,MamGypsy2-LTR,sum_abs_mean_activity_score,0.1170,NA,0.1170,-0.0724,0.0724,0.0006,0.0006,0.0440,0.0440
4,global,4,MER52A,sum_abs_mean_activity_score,0.1055,NA,0.1055,-0.0660,0.0660,0.0216,0.0216,0.0179,0.0179
5,global,5,MamRep488,sum_abs_mean_activity_score,0.1035,NA,0.1035,-0.0619,0.0619,0.0417,0.0417,0.0000,0.0000
6,global,6,LTR88a,sum_abs_mean_activity_score,0.0928,NA,0.0928,-0.0609,0.0609,0.0221,0.0221,0.0098,0.0098
7,global,7,THE1B-int,sum_abs_mean_activity_score,0.0905,NA,0.0905,-0.0488,0.0488,0.0368,0.0368,0.0049,0.0049
8,global,8,MamGypLTR1a,sum_abs_mean_activity_score,0.0904,NA,0.0904,-0.0582,0.0582,0.0274,0.0274,0.0048,0.0048
9,global,9,Charlie17a,sum_abs_mean_activity_score,0.0897,NA,0.0897,-0.0544,0.0544,0.0274,0.0274,0.0079,0.0079


---
## 2 · TE-Expression Ranking

`E_TE-Seq_prep_results.tsv` has genes and repeat rows. Here I use the repeat rows only.

- keep `gene_or_te == "repeat"`;
- parse repeat-locus IDs in `gene_id` to TE-subfamily names;
- average expression across replicates for `PRO`, `ESEN`, `LSEN`;
- resolve duplicate TE names with `Max Sum` on those condition means.

The global expression score is the sum of the three condition means. Condition-specific lists use the matching mean expression.

In [31]:
# TE universe + TE-Seq repeat rows.
reference_te_subfamilies <- load_reference_te_subfamilies(
  N_path = N_path,
  activity_scores = A
)

te_expression <- NULL
te_expression_condition_means <- NULL
te_expression_errors <- character()

for (candidate_idx in seq_along(te_expression_path_resolved)) {
  candidate_path <- te_expression_path_resolved[[candidate_idx]]
  candidate_label <- te_expression_path_labels[[candidate_idx]]
  if (!file.exists(candidate_path)) {
    te_expression_errors <- c(te_expression_errors, sprintf("missing: %s", candidate_label))
    next
  }

  attempt <- tryCatch(
    load_te_expression_condition_means(
      E_path = candidate_path,
      reference_te_subfamilies = reference_te_subfamilies,
      duplicate_method = te_duplicate_method,
      condition_groups = condition_groups[condition_order],
      condition_order = condition_order,
      verbose = FALSE
    ),
    error = function(e) e
  )

  if (!inherits(attempt, "error")) {
    te_expression <- attempt
    te_expression_condition_means <- te_expression$condition_means
    te_expression_path <- candidate_path
    te_expression_path_label <- candidate_label
    break
  }

  te_expression_errors <- c(
    te_expression_errors,
    sprintf("%s: %s", candidate_label, conditionMessage(attempt))
  )
}

if (is.null(te_expression)) {
  te_expression_selection_summary <- data.frame(
    te_expression_path = paste(te_expression_path_labels, collapse = "; "),
    status = "skipped_no_te_subfamily_rows",
    reason = paste(te_expression_errors, collapse = " | "),
    stringsAsFactors = FALSE
  )
} else {
  te_expression_selection_summary <- te_expression$selection_summary
  te_expression_selection_summary$te_expression_path <- te_expression_path_label
  te_expression_selection_summary$status <- "ok"
}

te_expression_selection_summary


expression_rows_total,expression_rows_kept_as_te,rows_kept_by_reference,rows_kept_by_type_column,te_type_column,unique_te_subfamilies_before_duplicate_handling,duplicate_groups_before,redundant_rows_before,te_subfamilies_after_duplicate_handling,duplicate_groups_after,duplicate_method,condition_mean_step,te_expression_path,status
<int>,<int>,<int>,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
1682318,1596502,1397700,1596502,gene_or_te,6151,6151,1590351,6151,0,max_sum,before_duplicate_handling,data/Hg38/E_TE-Seq_prep_results.tsv,ok


In [32]:
# TE expression rankings after condition means and duplicate handling.
if (is.null(te_expression_condition_means)) {
  te_expression_ranked <- data.frame(
    scope = character(),
    rank = integer(),
    te_subfamily = character(),
    ranking_metric = character(),
    ranking_score = numeric(),
    stringsAsFactors = FALSE
  )
  expression_ranked_by_scope <- list()
  te_expression_top_k <- data.frame(
    source = character(),
    scope = character(),
    k = integer(),
    rank_within_k = integer(),
    rank = integer(),
    te_subfamily = character(),
    ranking_metric = character(),
    ranking_score = numeric(),
    stringsAsFactors = FALSE
  )
  message("Skipping TE-expression ranking until the TE-Seq expression table is available.")
} else {
  expression_ranked_by_scope <- stats::setNames(lapply(ranking_scopes, function(scope) {
    rank_te_expression_condition_means(
      te_expression_condition_means = te_expression_condition_means,
      k_values = k_values,
      condition_order = condition_order,
      scope = scope
    )
  }), ranking_scopes)
  te_expression_ranked <- expression_ranked_by_scope$global
  te_expression_top_k <- do.call(rbind, lapply(expression_ranked_by_scope, function(ranked_table) {
    top_k_long_table(
      ranked_table = ranked_table,
      k_values = k_values,
      source = "te_expression",
      score_col = "ranking_score"
    )
  }))
  rownames(te_expression_top_k) <- NULL
}

te_expression_ranked_display <- te_expression_ranked
if (nrow(te_expression_ranked_display) > 0L) {
  numeric_cols <- vapply(te_expression_ranked_display, is.numeric, logical(1))
  te_expression_ranked_display[numeric_cols] <- lapply(te_expression_ranked_display[numeric_cols], round, digits = 4)
}
head(te_expression_ranked_display, 20)


,scope,rank,te_subfamily,ranking_metric,ranking_score,sum_condition_mean_expression_score,average_condition_mean_expression,mean_expression_PRO,mean_expression_ESEN,mean_expression_LSEN
,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,global,1,6kbHsap,sum_condition_mean_expression_score,44246.92,44246.92,14748.974,22535.033,9471.640,12240.248
2,global,2,AluSx,sum_condition_mean_expression_score,40981.45,40981.45,13660.482,17199.785,7878.480,15903.182
3,global,3,(CTGCTG)n,sum_condition_mean_expression_score,39749.08,39749.08,13249.692,15430.722,13569.854,10748.499
4,global,4,AluSz,sum_condition_mean_expression_score,39369.47,39369.47,13123.156,19719.569,8686.096,10963.803
5,global,5,GA-rich,sum_condition_mean_expression_score,36278.37,36278.37,12092.790,12456.067,11579.229,12243.072
6,global,6,FRAM,sum_condition_mean_expression_score,29575.48,29575.48,9858.493,12022.365,5265.815,12287.299
7,global,7,(TCTGAG)n,sum_condition_mean_expression_score,27385.53,27385.53,9128.510,14526.097,5227.328,7632.105
8,global,8,(CAACCG)n,sum_condition_mean_expression_score,26895.46,26895.46,8965.152,14302.398,5543.835,7049.223
9,global,9,(CCCTCTT)n,sum_condition_mean_expression_score,24284.43,24284.43,8094.809,1796.817,12122.437,10365.174


---
## 3 · Top-k Intersections

Same `k` values for both lists. The summary includes the global list and the three condition-specific lists.

The short global top 10 below is the shared TE set ranked by the sum of activity rank and expression rank.

In [33]:
# Top-k overlaps: global and condition-specific.
if (nrow(te_expression_ranked) == 0L) {
  intersection_summary <- data.frame(
    scope = rep(ranking_scopes, each = length(k_values)),
    k = rep(k_values, times = length(ranking_scopes)),
    n_activity_top = NA_integer_,
    n_te_expression_top = NA_integer_,
    n_raw_top = NA_integer_,
    n_intersection = NA_integer_,
    pct_activity_top = NA_real_,
    pct_te_expression_top = NA_real_,
    pct_raw_top = NA_real_,
    jaccard = NA_real_,
    intersect_te_subfamilies = NA_character_,
    status = "skipped_missing_te_expression_table",
    stringsAsFactors = FALSE
  )
  intersection_members <- data.frame(
    scope = character(),
    k = integer(),
    te_subfamily = character(),
    activity_rank = integer(),
    activity_score = numeric(),
    te_expression_rank = integer(),
    te_expression_score = numeric(),
    raw_rank = integer(),
    raw_expression_sum = numeric(),
    combined_rank_sum = integer(),
    stringsAsFactors = FALSE
  )
  global_top10_intersection <- data.frame(
    global_intersection_rank = integer(),
    scope = character(),
    te_subfamily = character(),
    activity_rank = integer(),
    activity_score = numeric(),
    te_expression_rank = integer(),
    te_expression_score = numeric(),
    raw_rank = integer(),
    raw_expression_sum = numeric(),
    combined_rank_sum = integer(),
    stringsAsFactors = FALSE
  )
  condition_top10_intersections <- global_top10_intersection[0, , drop = FALSE]
} else {
  intersection_objects <- lapply(ranking_scopes, function(scope) {
    build_top_k_intersections(
      activity_ranked = activity_ranked_by_scope[[scope]],
      raw_ranked = expression_ranked_by_scope[[scope]],
      k_values = k_values,
      raw_score_col = "ranking_score",
      scope = scope
    )
  })
  intersection_summary <- do.call(rbind, lapply(intersection_objects, `[[`, "summary"))
  intersection_members <- do.call(rbind, lapply(intersection_objects, `[[`, "members"))

  global_top10_intersection <- build_global_intersection_top_n(
    activity_ranked = activity_ranked_by_scope$global,
    raw_ranked = expression_ranked_by_scope$global,
    top_n = 10L,
    raw_score_col = "ranking_score",
    scope = "global"
  )
  condition_top10_intersections <- do.call(rbind, lapply(condition_order, function(scope) {
    build_global_intersection_top_n(
      activity_ranked = activity_ranked_by_scope[[scope]],
      raw_ranked = expression_ranked_by_scope[[scope]],
      top_n = 10L,
      raw_score_col = "ranking_score",
      scope = scope
    )
  }))

  if (is.null(condition_top10_intersections)) {
    condition_top10_intersections <- global_top10_intersection[0, , drop = FALSE]
  }

  if (nrow(global_top10_intersection) == 0L) {
    message("No shared TE subfamilies in the global top-100 activity/expression lists.")
  }

  for (scope in ranking_scopes) {
    for (k in k_values) {
      cat(sprintf("\n%s top %s overlap\n", scope, k))
      display_intersect(
        utils::head(activity_ranked_by_scope[[scope]]$te_subfamily, min(k, nrow(activity_ranked_by_scope[[scope]]))),
        utils::head(expression_ranked_by_scope[[scope]]$te_subfamily, min(k, nrow(expression_ranked_by_scope[[scope]]))),
        name_E = "cis-regulatory activity",
        name_N = "TE expression",
        title = sprintf("%s top %s TE-subfamily overlap", scope, k),
        plot = FALSE
      )
    }
  }
}

intersection_summary



global top 3 overlap
cis-regulatory activity genes: 3
TE expression genes: 3
amount of genes we keep in cis-regulatory activity: 0 (0.00% of cis-regulatory activity)
amount of genes we keep in TE expression: 0 (0.00% of TE expression)
Common genes: 0 (0.00% of union)
Only in cis-regulatory activity: 3 (100.00% of cis-regulatory activity, 50.00% of union)
Only in TE expression: 3 (100.00% of TE expression, 50.00% of union)

global top 5 overlap
cis-regulatory activity genes: 5
TE expression genes: 5
amount of genes we keep in cis-regulatory activity: 0 (0.00% of cis-regulatory activity)
amount of genes we keep in TE expression: 0 (0.00% of TE expression)
Common genes: 0 (0.00% of union)
Only in cis-regulatory activity: 5 (100.00% of cis-regulatory activity, 50.00% of union)
Only in TE expression: 5 (100.00% of TE expression, 50.00% of union)

global top 10 overlap
cis-regulatory activity genes: 10
TE expression genes: 10
amount of genes we keep in cis-regulatory activity: 0 (0.00% of c

scope,k,n_activity_top,n_te_expression_top,n_raw_top,n_intersection,pct_activity_top,pct_te_expression_top,pct_raw_top,jaccard,intersect_te_subfamilies
<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
global,3,3,3,3,0,0,0,0,0.00000000,
global,5,5,5,5,0,0,0,0,0.00000000,
global,10,10,10,10,0,0,0,0,0.00000000,
global,20,20,20,20,0,0,0,0,0.00000000,
global,50,50,50,50,2,4,4,4,0.02040816,"Tigger10, MER91A"
global,100,100,100,100,7,7,7,7,0.03626943,"Charlie17a, Tigger10, MER91A, L1PA3, LTR82A, HSMAR1, FAM"
PRO,3,3,3,3,0,0,0,0,0.00000000,
PRO,5,5,5,5,0,0,0,0,0.00000000,
PRO,10,10,10,10,0,0,0,0,0.00000000,


In [34]:
# Short shared list, ranked by combined activity/expression rank.
global_top10_intersection


global_intersection_rank,scope,te_subfamily,activity_rank,activity_score,te_expression_rank,te_expression_score,raw_rank,raw_expression_sum,combined_rank_sum
<int>,<chr>,<chr>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>
1,global,Tigger10,19,0.07085613,37,7572.467,37,7572.467,56
2,global,MER91A,25,0.06142058,44,6541.660,44,6541.660,69
3,global,Charlie17a,9,0.08972896,71,4595.346,71,4595.346,80
4,global,L1PA3,39,0.04940584,68,4737.364,68,4737.364,107
5,global,HSMAR1,64,0.03769983,45,6489.598,45,6489.598,109
6,global,LTR82A,62,0.03964357,72,4589.127,72,4589.127,134
7,global,FAM,77,0.03289546,75,4454.986,75,4454.986,152


In [35]:
# Same short list by condition, useful when the global overlap is sparse.
condition_top10_intersections


global_intersection_rank,scope,te_subfamily,activity_rank,activity_score,te_expression_rank,te_expression_score,raw_rank,raw_expression_sum,combined_rank_sum
<int>,<chr>,<chr>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>
1,PRO,Tigger10,25,0.040786451,64,1596.173,64,1596.173,89
2,PRO,MER91A,26,0.040392804,99,1006.209,99,1006.209,125
3,PRO,HSMAR1,73,0.023791854,63,1615.889,63,1615.889,136
4,PRO,X9_LINE,50,0.029186140,88,1223.005,88,1223.005,138
1,ESEN,Tigger10,7,0.030069681,32,3144.499,32,3144.499,39
2,ESEN,HSMAR1,35,0.013907979,37,2771.536,37,2771.536,72
3,ESEN,MER91A,57,0.011010852,20,4989.846,20,4989.846,77
4,ESEN,Charlie17a,12,0.027382901,67,1770.814,67,1770.814,79
5,ESEN,LTR82A,65,0.009610220,54,2127.346,54,2127.346,119


---
## 4 · Export Tables

TSVs go into `results/<run>/exploratory/most_active_te_subfamilies/`.

In [36]:
if (nrow(te_expression_ranked) == 0L && !isTRUE(write_incomplete_outputs)) {
  saved_outputs <- data.frame(
    name = "exports_skipped",
    format = NA_character_,
    path = NA_character_,
    reason = "TE-expression table missing; set write_incomplete_outputs <- TRUE to write skip-status tables",
    stringsAsFactors = FALSE
  )
} else {
  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

  saved_outputs <- rbind(
    write_result_object(
      input_dir = output_dir,
      name = "cis_regulatory_activity_top_k_te_subfamilies",
      object = activity_top_k,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "te_expression_extraction_summary",
      object = te_expression_selection_summary,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "raw_te_extraction_summary",
      object = te_expression_selection_summary,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "te_expression_top_k_te_subfamilies",
      object = te_expression_top_k,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "top_k_intersections",
      object = intersection_summary,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "top_k_intersection_members",
      object = intersection_members,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "global_top10_intersection_te_subfamilies",
      object = global_top10_intersection,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    ),
    write_result_object(
      input_dir = output_dir,
      name = "condition_top10_intersection_te_subfamilies",
      object = condition_top10_intersections,
      write_tsv = TRUE,
      write_rds = FALSE,
      overwrite = TRUE
    )
  )
}

saved_outputs


name,format,path
<chr>,<chr>,<chr>
cis_regulatory_activity_top_k_te_subfamilies,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/cis_regulatory_activity_top_k_te_subfamilies.tsv
te_expression_extraction_summary,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/te_expression_extraction_summary.tsv
raw_te_extraction_summary,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/raw_te_extraction_summary.tsv
te_expression_top_k_te_subfamilies,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/te_expression_top_k_te_subfamilies.tsv
top_k_intersections,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/top_k_intersections.tsv
top_k_intersection_members,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/top_k_intersection_members.tsv
global_top10_intersection_te_subfamilies,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/global_top10_intersection_te_subfamilies.tsv
condition_top10_intersection_te_subfamilies,tsv,/Users/vinzent/craTEs/CraTEs_Senescence_Analysis/results/results_E_Senescene_TE_Seq_raw_N_c_100_Hg38_craTEs/exploratory/most_active_te_subfamilies/condition_top10_intersection_te_subfamilies.tsv


---
## 5 · Interpretation

TE subfamilies in the overlap are follow-up candidates. They are active in the cis-regulatory model and also visible in the TE-expression signal.

This does not prove promoter function. It gives a short list for promoter-focused checks next to the cis-regulatory activity call: locus context, TSS proximity, strand-aware TE signal, chromatin marks, or targeted validation.